In [2]:
import pandas as pd
import numpy as np

In [ ]:
class Equity:
    
    #Risk weight table for Delta
    delta_risk_weights_spot = {1: 0.55, 2: 0.6, 3: 0.45, 4: 0.55, 5: 0.3, 6: 0.35, 7: 0.4, 8: 0.5, 9: 0.7, 10: 0.5, 11: 0.7, 12: 0.15, 13: 0.25}
    delta_risk_weights_repo = {keys: values / 100 for keys, values in delta_risk_weights_spot.items()}
    
    #Within bucket correlation mapped by bucket number - delta
    intra_corr_bucket_delta = {1: 0.15, 2: 0.15, 3: 0.15, 4: 0.15, 5: 0.25, 6: 0.25, 7: 0.25, 8: 0.25, 9: 0.075, 10: 0.125, 11: 0., 12: 0.8, 13: 0.8}
    
    #Across bucket correlation mapped by bucket number - delta
    across_corr_delta = np.full((13, 13), 0.45)
    
    rows, cols = np.indices(across_corr_delta.shape)

    #index = bucket number - 1
    across_corr_delta[(rows <= 9) & (cols <= 9)] = 0.15
    across_corr_delta[(rows == 10) | (cols == 10)] = 0.
    across_corr_delta[((rows == 11) & (cols == 12)) | ((rows == 12) & (cols == 11))] = 0.75
    
    def __init__(self, data):
        #Assume the data has already been cleansed, filtered and grouped
        self.original_data = data 
        self.data = data.copy()
        
    def weight_sensitivities(self):
        
        temp_risk_weights = np.select(  [
                                        self.data["Sensi Type"] == "Delta", 
                                        self.data["Sensi Type"] == "Vega"
                                        ], 
                                        [
                                        np.select([ self.data["Price Type"] == "Spot", 
                                                    self.data["Price Type"] == "Repo"
                                                    ],
                                                    [
                                                    self.data["SA Bucket"].map(Equity.delta_risk_weights_spot), 
                                                    self.data["SA Bucket"].map(Equity.delta_risk_weights_repo)
                                                    ]), 
                                        self.data["SA Bucket"].isin([9, 10, 11]).map(   {
                                                                                        True: np.minimum(0.55 * np.sqrt(60/10), 1), 
                                                                                        False: np.minimum(0.55 * np.sqrt(20/10), 1)
                                                                                        })],
                                        default=1)
        
        self.data["Weighted Sensitivities"] = self.data["Sensitivity (reporting currency equiv.)"] * temp_risk_weights
        
    def perform_intra_bucket_aggregation(self, sensi_type):
        #Filter data to only include targeted sensitivity type
        filtered_data = self.data[self.data["Sensi Type"] == sensi_type]
        
        #Aggregate data of the same sensitivities
        if sensi_type == "Delta":
            cols_to_grp = ["SA Bucket", "Issuer", "Price Type"]
        elif sensi_type == "Vega":
            cols_to_grp = ["SA Bucket", "Tenor"]
        else:
            cols_to_grp = ["SA Bucket", "Issuer", "CVR+/CVR-"]
        
        filtered_data = filtered_data.groupby(cols_to_grp)["Weighted Sensitivities"].sum().reset_index()
        
        #Pivot in case of CVR to have separate columns CVR+ and CVR-
        if sensi_type == "Curvature":
            filtered_data = filtered_data.pivot_table(index=["SA Bucket", "Issuer"], columns="CVR+/CVR-", values="Weighted Sensitivities", aggfunc="sum").reset_index()
            
            #Initiate variables to store CVR and K for each bucket for curvature
            temp_CVR_plus_by_bucket = [[], [], []]   #[[medium], [high], [low]]
            temp_CVR_minus_by_bucket = [[], [], []]   #[[medium], [high], [low]]
            temp_Kb_minus_by_bucket = [[], [], []]   #[[medium], [high], [low]]
            
        #Initiate variables to store K for each bucket
        temp_Kb_by_bucket = [[], [], []]   #[[medium], [high], [low]]
        K_by_bucket = []
            
        #Create a list of unique buckets
        bucket_list = filtered_data["SA Bucket"].drop_duplicates()
        
        for bucket in bucket_list:
            #Filter data for each bucket
            bucket_data = filtered_data[filtered_data["SA Bucket"] == bucket].reset_index(drop=True)

            #Initiate variables to store correlation matrix for each bucket
            corr_matrix_by_bucket = dict.fromkeys(bucket_list, None)
            
            n = len(bucket_data)
            
            #Generate correlation matrix for each bucket
            temp_corr_matrix = np.full((n, n), Equity.intra_corr_bucket_delta.get(bucket))

            if sensi_type != "Curvature":
                
                #Aggregate directly for bucket 11 MAR21.79(1)
                if bucket == 11:
                    for i in range(3):
                        temp_Kb_by_bucket[i].append(np.absolute(bucket_data["Weighted Sensitivities"]).sum())
                    continue
                
                np.fill_diagonal(temp_corr_matrix, 1)
                
                for i in range(n):
                    for j in range(n):
                        if i != j:
                            
                            if sensi_type == "Delta":
                                #Check if one sensi is spot and the other is repo
                                if bucket_data.loc[i, "Price Type"] != bucket_data.loc[j, "Price Type"]:
                                    
                                    #Check if the issuer is the same
                                    if bucket_data.loc[i, "Issuer"] == bucket_data.loc[j, "Issuer"]:
                                        
                                        #MAR21.78(1)
                                        temp_corr_matrix[i, j] = 0.999
                                        
                                    else:
                                        
                                        #MAR21.78(4)
                                        temp_corr_matrix[i, j] *= 0.999

                            else: #Vega
                                
                                #Check option tenor
                                temp_corr_matrix[i, j] *= np.exp(
                                                            -0.01 * np.abs(bucket_data.loc[i, "Tenor"] - bucket_data.loc[j, "Tenor"]) / 
                                                            np.minimum(bucket_data.loc[i, "Tenor"], bucket_data.loc[j, "Tenor"]))
                                                
                                temp_corr_matrix[i, j] = np.minimum(temp_corr_matrix[i, j], 1)
                                    
                #High and low scenario
                temp_high_corr_matrix = np.minimum(1.25 * temp_corr_matrix, 1)
                temp_low_corr_matrix = np.maximum(2 * temp_corr_matrix - 1, 0.75 * temp_corr_matrix)
            
                #Store results
                corr_matrix_by_bucket[bucket] = [temp_corr_matrix, temp_high_corr_matrix, temp_low_corr_matrix]
                
                #Calculate sum of weighted sensitivities of the bucket
                weighted_sensi = np.asarray(bucket_data["Weighted Sensitivities"])
                
                #Aggregate
                for i in range(3):
                    temp_Kb_by_bucket[i].append(np.sqrt(np.maximum(0, np.dot(np.transpose(weighted_sensi),np.dot(weighted_sensi, corr_matrix_by_bucket[bucket][i])))))
            
            else:
                
                #Aggregate directly for bucket 16 MAR21.79(2)
                if bucket == 11:
                    for i in range(3):
                        temp_CVR_plus_by_bucket[i].append(bucket_data["CVR+"].sum())
                        temp_CVR_minus_by_bucket[i].append(bucket_data["CVR-"].sum())
                        temp_Kb_by_bucket[i].append(np.maximum(bucket_data["CVR+"], 0).sum())
                        temp_Kb_minus_by_bucket[i].append(np.maximum(bucket_data["CVR-"], 0).sum())
                    continue
                
                psi_matrix_CVR_plus = np.zeros((n, n))
                psi_matrix_CVR_minus = np.zeros((n, n))                
                
                for i in range(n):
                    for j in range(n):
                        if i != j:
                            #Calculate psi matrix for CVR+ and CVR-
                            if bucket_data.loc[i, "CVR+"] >= 0 or bucket_data.loc[j, "CVR+"] >= 0:
                                psi_matrix_CVR_plus[i, j] = 1
                            
                            if bucket_data.loc[i, "CVR-"] >= 0 or bucket_data.loc[j, "CVR-"] >= 0:
                                psi_matrix_CVR_minus[i, j] = 1
                            
                #correlation matrix = delta matrix ** 2
                np.fill_diagonal(temp_corr_matrix, 0)
                temp_corr_matrix = temp_corr_matrix ** 2
                temp_high_corr_matrix = np.minimum(1.25 * temp_corr_matrix, 1)
                temp_low_corr_matrix = np.maximum(2 * temp_corr_matrix - 1, 0.75 * temp_corr_matrix)
                
                #Store results
                corr_matrix_by_bucket[bucket] = [temp_corr_matrix, temp_high_corr_matrix, temp_low_corr_matrix]

                #Calculate K for the bucket
                for i in range(3):
                    temp_CVR_plus_by_bucket[i].append(bucket_data["CVR+"].sum())
                    
                    temp_CVR_minus_by_bucket[i].append(bucket_data["CVR-"].sum())
                    
                    temp_Kb_by_bucket[i].append(np.sqrt(
                                                np.maximum(0, 
                                                np.square(np.maximum(0, bucket_data["CVR+"])).sum() + 
                                                np.dot(bucket_data["CVR+"], np.dot(np.transpose(bucket_data["CVR+"]), np.multiply(corr_matrix_by_bucket[bucket][i], psi_matrix_CVR_plus)))
                                                )))
                    
                    temp_Kb_minus_by_bucket[i].append(np.sqrt(
                                                np.maximum(0, 
                                                np.square(np.maximum(0, bucket_data["CVR-"])).sum() + 
                                                np.dot(bucket_data["CVR-"], np.dot(np.transpose(bucket_data["CVR-"]), np.multiply(corr_matrix_by_bucket[bucket][i], psi_matrix_CVR_minus)))
                                                )))
                
        if sensi_type != "Curvature":
            
            for i in range(3):
                K_by_bucket.append(pd.DataFrame({"SA Bucket": bucket_list, "K": temp_Kb_by_bucket[i]}).reset_index(drop=True))
                
        else:
            
            for i in range(3):
                K_by_bucket.append(pd.DataFrame({"SA Bucket": bucket_list,
                                                "CVR+": temp_CVR_plus_by_bucket[i],
                                                "CVR-": temp_CVR_minus_by_bucket[i],
                                                "K+": temp_Kb_by_bucket[i], 
                                                "K-": temp_Kb_minus_by_bucket[i], 
                                                "K": np.maximum(temp_Kb_by_bucket[i], temp_Kb_minus_by_bucket[i])})
                                                .reset_index(drop=True))
        
        return K_by_bucket
    
    def perform_across_bucket_aggregation(self, sensi_type, intra_agg_result):
        
        capital_charge = []
        
        if sensi_type != "Curvature":
            
            filtered_data = self.data[self.data["Sensi Type"] == sensi_type]
            
            #Calculate Sb according to formulation in MAR21.4(5)
            Sb_by_bucket = filtered_data.groupby("SA Bucket")["Weighted Sensitivities"].sum().reset_index()
            
            #Generate the correlation matrix
            n = len(Sb_by_bucket)
            medium_corr_matrix = np.zeros((n, n))
            for i in range(n):
                for j in range(n):
                    if i != j:
                        medium_corr_matrix = Equity.across_corr_delta[Sb_by_bucket.loc[i, "SA Bucket"] - 1, Sb_by_bucket.loc[j, "SA Bucket"] - 1] #Bucket number -1 since bucket numbers start with 1 and python start counts from 0
            
            high_corr_matrix = np.minimum(1.25 * medium_corr_matrix, 1)
            low_corr_matrix = np.maximum(2 * medium_corr_matrix - 1, 0.75 * medium_corr_matrix)
            
            corr_matrices = [medium_corr_matrix, high_corr_matrix, low_corr_matrix]
            
            #Calculate the capital charge under each scenario [medium, high, low]
            for i in range(3):
                Kb = intra_agg_result[i]["K"]
                Sb = Sb_by_bucket["Weighted Sensitivities"]
                capital_charge.append(np.sqrt(
                                        np.square(Kb).sum() + 
                                        np.dot(np.transpose(Sb), np.dot(Sb, corr_matrices[i]))
                                        ))
                
                #For each scenario, we have to check whether an alternative formulation for Sb is required as stipulated in MAR21.4(5)
                if capital_charge[i] < 0:
                    Sb_Alt = np.maximum(np.minimum(Sb, Kb), -1 * Kb)
                    capital_charge[i] = (np.sqrt(
                                            np.square(Kb).sum() + 
                                            np.dot(np.transpose(Sb_Alt),np.dot(Sb_Alt, corr_matrices[i]))
                                            ))
                    
        else:
            
            temp_K_by_bucket = intra_agg_result[0]
            n = len(temp_K_by_bucket)
            
            #Generate the correlation matrix
            medium_corr_matrix = np.zeros((n, n))
            for i in range(n):
                for j in range(n):
                    if i != j:
                        medium_corr_matrix = Equity.across_corr_delta[temp_K_by_bucket.loc[i, "SA Bucket"] - 1, temp_K_by_bucket.loc[j, "SA Bucket"] - 1] ** 2 #Bucket number -1 since bucket numbers start with 1 and python start counts from 0
            
            high_corr_matrix = np.minimum(1.25 * medium_corr_matrix, 1)
            low_corr_matrix = np.maximum(2 * medium_corr_matrix - 1, 0.75 * medium_corr_matrix)
            
            corr_matrices = [medium_corr_matrix, high_corr_matrix, low_corr_matrix]
            
            Sb = [[], [], []] #[[medium], [high], [low]]
            psi_matrix = [[], [], []] #[[medium], [high], [low]]
            
            for i in range(3):
                temp_K_by_bucket = intra_agg_result[i]
                
                for j in range(n):                
                    #Calculate Sb according to formulation in MAR21.5(4)
                    if temp_K_by_bucket["K"][j] == temp_K_by_bucket["K+"][j]:
                        Sb[i].append(temp_K_by_bucket["CVR+"][j])
                    elif temp_K_by_bucket["K"][j] == temp_K_by_bucket["K-"][j]:
                        Sb[i].append(temp_K_by_bucket["CVR-"][j])
                    elif temp_K_by_bucket["CVR+"][j] > temp_K_by_bucket["CVR-"][j]:
                        Sb[i].append(temp_K_by_bucket["CVR+"][j])
                    else:
                        Sb[i].append(temp_K_by_bucket["CVR-"][j])
            
                #Generate the psi matrix according to MAR21.5(4)(b)
                psi_matrix[i] = np.zeros((n, n))
                for k in range(n):
                    for l in range(n):
                        if Sb[i][k] >= 0 or Sb[i][l] >= 0:
                            psi_matrix[i][k, l] = 1
                
                #Store results                
                intra_agg_result[i]["Sb"] = Sb[i]
            
                #Calculate capital charge
                capital_charge.append(np.sqrt(
                                        np.maximum(0, 
                                        np.square(intra_agg_result[i]["K"]).sum() + 
                                        np.dot(Sb[i], np.dot(np.transpose(Sb[i]), np.multiply(corr_matrices[i], psi_matrix[i])))
                                        )))
        
        result = pd.DataFrame({"Medium": [capital_charge[0]], "High": [capital_charge[1]], "Low": [capital_charge[2]]}, index=[sensi_type])
        
        return result
            

In [ ]:
#For quick test
test = pd.read_excel("Data/temp_frtb_data.xlsx", "Equity")

x = Equity(test)
x.weight_sensitivities()
intra = x.perform_intra_bucket_aggregation("Curvature")
x.perform_across_bucket_aggregation("Curvature", intra)

,Medium,High,Low
CVR,98699.659383,98514.840457,98884.132874
